# Problem 96

<p>Su Doku (Japanese meaning <i>number place</i>) is the name given to a popular puzzle concept. Its origin is unclear, but credit must be attributed to Leonhard Euler who invented a similar, and much more difficult, puzzle idea called Latin Squares. The objective of Su Doku puzzles, however, is to replace the blanks (or zeros) in a 9 by 9 grid in such that each row, column, and 3 by 3 box contains each of the digits 1 to 9. Below is an example of a typical starting puzzle grid and its solution grid.</p>
<div class="center">
<img src="resources/images/0096_1.png?1678992052" alt="0096_1.png">     <img src="resources/images/0096_2.png?1678992052" alt="0096_2.png">
</div>
<p>A well constructed Su Doku puzzle has a unique solution and can be solved by logic, although it may be necessary to employ "guess and test" methods in order to eliminate options (there is much contested opinion over this). The complexity of the search determines the difficulty of the puzzle; the example above is considered <i>easy</i> because it can be solved by straight forward direct deduction.</p>
<p>The 6K text file, <a href="resources/documents/0096_sudoku.txt">sudoku.txt</a> (right click and 'Save Link/Target As...'), contains fifty different Su Doku puzzles ranging in difficulty, but all with unique solutions (the first puzzle in the file is the example above).</p>
<p>By solving all fifty puzzles find the sum of the 3-digit numbers found in the top left corner of each solution grid; for example, 483 is the 3-digit number found in the top left corner of the solution grid above.</p>

In [4]:
# read in grids
# create sudoku solver function
# make into actual arrays for easier manipulation

# fill in last in row/column/box
# block row/column/box for each number, check if only one 0 remains in other boxes

import numpy as np

def read_grid(file_path):
    grids = {}
    with open(file_path, 'r') as f:
        lines = [line.strip() for line in f.readlines() if line.strip()]

    i = 0
    while i < len(lines):
        if lines[i].startswith('Grid'):
            grid_name = lines[i]
            rows = lines[i+1:i+10]
            grid = np.array([[int(num) for num in row] for row in rows])
            grids[grid_name] = grid
            i += 10  # Move to the next grid
        else:
            i += 1  # Skip any unexpected lines
    return grids


def get_box_coords(r, c): # returns the coordinates of the 3x3 box that contains the cell at (r, c)
    box_row = (r // 3) * 3
    box_col = (c // 3) * 3
    return [(box_row + i, box_col + j) for i in range(3) for j in range(3)]


def get_units(): # returns pieces (sets) of the sudoku grid that are used to check for validity
    units = []

    # 9 row units
    for r in range(9):
        units.append([(r, c) for c in range(9)])

    # 9 column units
    for c in range(9):
        units.append([(r, c) for r in range(9)])

    # 9 box units
    for box_r in range(0, 9, 3):
        for box_c in range(0, 9, 3):
            units.append(get_box_coords(box_r, box_c))

    return units


def get_candidates(grid, r, c): # returns a set of possible numbers at (r,c)
    if grid[r, c] != 0:
        return set() # Already filled

    used = set(grid[r, :])
    used |= set(grid[:, c]) # |= union 
    box_coords = get_box_coords(r, c)
    used |= {grid[br, bc] for br, bc in box_coords} # iterates through box

    return set(range(1, 10)) - used


def naked_singles(grid): # fills in cells with only one candidate
    progress = False
    for r in range(9):
        for c in range(9):
            if grid[r, c] == 0:
                candidates = get_candidates(grid, r, c)
                if len(candidates) == 1:
                    grid[r, c] = candidates.pop()
                    progress = True
    return progress


def hidden_singles(grid): # fills in cells where a number can only go in one place in a unit
    progress = False
    for unit in get_units():
        values_in_unit = {grid[cell] for cell in unit if grid[cell] != 0}
        for num in range(1, 10):
            if num in values_in_unit:
                continue
            candidates = [cell for cell in unit if grid[cell] == 0 and num in get_candidates(grid, *cell)]
            if len(candidates) == 1:
                grid[candidates[0]] = num
                progress = True
    return progress


def is_solved(grid):
    return np.all(grid != 0)


def propagate(grid): # repeatedly applies naked and hidden singles until no further progress can be made
    while True:
        progress1 = naked_singles(grid)
        progress2 = hidden_singles(grid)
        if not (progress1 or progress2):
            break


def find_best_cell(grid): # finds the cell with the fewest candidates
    best_cell = None
    best_candidates = None

    for r in range(9):
        for c in range(9):
            if grid[r, c] == 0:
                candidates = get_candidates(grid, r, c)
                if best_candidates is None or len(candidates) < len(best_candidates): # check if less candidates than current best
                    best_cell = (r, c)
                    best_candidates = candidates

    if best_cell is None:
        return None

    return best_cell[0], best_cell[1], best_candidates # splits the tuple into row, column, and candidates


def solve(grid): # main solving function, uses backtracking
    grid = grid.copy()  # Work on a copy to avoid modifying the original grid
    propagate(grid)

    if is_solved(grid):
        return grid

    result = find_best_cell(grid)
    r, c, candidates = result

    if len(candidates) == 0:
        return None  # No candidates left, backtrack

    for num in candidates:
        attempt = grid.copy()
        attempt[r, c] = num
        solution = solve(attempt)
        if solution is not None:
            return solution

    return None  # No solution found, backtrack

In [ ]:
grids = read_grid('0096_sudoku.txt')
header_sum = 0

for name, grid in grids.items():
    solution = solve(grid)
    if solution is not None:
        #print(f"{name} solved:\n{solution}\n")
        header_sum += int(''.join(map(str, solution[0, :3])))  # Sum the first three digits of the first row
    else:
        print(f"{name} has no solution.\n")

print(f"Header sum: {header_sum}")

Header sum: 24702


# Problem 97

The first known prime found to exceed one million digits was discovered in 1999, and is a Mersenne prime of the form $2^{6972593} - 1$; it contains exactly $2\,098\,960$ digits. Subsequently other Mersenne primes, of the form $2^p - 1$, have been found which contain more digits.
However, in 2004 there was found a massive non-Mersenne prime which contains $2\,357\,207$ digits: $28433 \times 2^{7830457} + 1$.
Find the last ten digits of this prime number.
